# BirdCLEF 2026 — Baseline Training (Self-Contained)

EfficientNet-B0 SED baseline. Everything inline — no external imports needed.

**Runtime**: T4 GPU → Runtime > Change runtime type > T4 GPU

In [ ]:
# === 1. Setup & Data ===
import os, subprocess, sys
!pip install -q torch torchaudio timm pytorch-lightning scikit-learn pandas numpy pyyaml kaggle
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_2d7b97262c459561596c3b49927c7d29'

DATA_DIR = '/content/data/birdclef-2026'
if not os.path.isdir(DATA_DIR):
    !kaggle competitions download birdclef-2026 -p /content/data
    !cd /content/data && unzip -q -o birdclef-2026.zip -d birdclef-2026
!ls {DATA_DIR}/

In [ ]:
# === 2. Imports & Data Loading ===
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import timm
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

AUDIO_DIR = f'{DATA_DIR}/train_audio'
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
train_df['primary_label'] = train_df['primary_label'].astype(str)
taxonomy = pd.read_csv(f'{DATA_DIR}/taxonomy.csv')
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str)
species_list = sorted(taxonomy['primary_label'].unique().tolist())
num_classes = len(species_list)
species_to_idx = {s: i for i, s in enumerate(species_list)}

print(f'Samples: {len(train_df)}, Species: {num_classes}')
print(f'Class distribution: {train_df["class_name"].value_counts().to_dict()}')

# Create folds grouped by author
train_df['fold'] = -1
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (_, val_idx) in enumerate(skf.split(train_df, train_df['primary_label'], train_df['author'])):
    train_df.loc[val_idx, 'fold'] = fold_idx
print(f'Fold distribution:\n{train_df["fold"].value_counts().sort_index()}')

In [ ]:
# === 3. Audio Utils ===
SR = 32000
DURATION = 5.0
SEG_SAMPLES = int(SR * DURATION)

def load_audio(path, sr=SR, max_dur=30):
    waveform, orig_sr = torchaudio.load(str(path))
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if orig_sr != sr:
        waveform = torchaudio.functional.resample(waveform, orig_sr, sr)
    max_samples = int(sr * max_dur)
    if waveform.shape[1] > max_samples:
        waveform = waveform[:, :max_samples]
    return waveform

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=2048, hop_length=512,
    n_mels=128, f_min=20, f_max=16000)
amp_to_db = torchaudio.transforms.AmplitudeToDB(top_db=80)

def to_mel(waveform):
    mel = mel_transform(waveform)
    mel = amp_to_db(mel)
    mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-8)
    return mel

# Test
sample_path = os.path.join(AUDIO_DIR, train_df['filename'].iloc[100])
w = load_audio(sample_path)
m = to_mel(w[:, :SEG_SAMPLES])
print(f'Audio: {w.shape}, Mel: {m.shape}')

In [ ]:
# === 4. Dataset ===
class BirdDataset(Dataset):
    def __init__(self, df, audio_dir, train=True):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, row['filename'])
        try:
            waveform = load_audio(path, SR, 30)
        except Exception:
            waveform = torch.zeros(1, SEG_SAMPLES)

        total = waveform.shape[1]
        if self.train:
            start = np.random.randint(0, max(1, total - SEG_SAMPLES + 1))
        else:
            start = 0
        seg = waveform[:, start:start + SEG_SAMPLES]
        if seg.shape[1] < SEG_SAMPLES:
            seg = F.pad(seg, (0, SEG_SAMPLES - seg.shape[1]))

        mel = to_mel(seg)

        target = torch.zeros(num_classes, dtype=torch.float32)
        label = str(row['primary_label'])
        if label in species_to_idx:
            target[species_to_idx[label]] = 1.0

        return mel, target

# Quick test
ds = BirdDataset(train_df.head(5), AUDIO_DIR, train=True)
mel, tgt = ds[0]
print(f'Mel: {mel.shape}, Target sum: {tgt.sum().item()}')

In [ ]:
# === 5. SED Model ===
class AttentionHead(nn.Module):
    def __init__(self, in_features, num_classes, hidden=512, dropout=0.5):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.att = nn.Linear(hidden, num_classes)
        self.cls = nn.Linear(hidden, num_classes)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = self.drop(F.relu(self.fc1(x)))
        att_w = torch.softmax(self.att(x), dim=1)
        frame_p = torch.sigmoid(self.cls(x))
        clip_p = (att_w * frame_p).sum(dim=1)
        return clip_p, frame_p

class SEDModel(nn.Module):
    def __init__(self, backbone='tf_efficientnet_b0_ns', nc=234):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=True,
            in_chans=1, num_classes=0, global_pool='')
        with torch.no_grad():
            feat = self.backbone(torch.randn(1, 1, 128, 312))
            feat_dim = feat.shape[1]
        self.pool = nn.AdaptiveAvgPool2d((None, 1))
        self.drop = nn.Dropout(0.25)
        self.head = AttentionHead(feat_dim, nc)

    def forward(self, x):
        f = self.backbone(x)
        f = self.pool(f).squeeze(-1).permute(0, 2, 1)
        f = self.drop(f)
        return self.head(f)

# Verify
model = SEDModel('tf_efficientnet_b0_ns', num_classes)
dummy = torch.randn(2, 1, 128, 312)
cp, fp = model(dummy)
print(f'clip_pred: {cp.shape}, frame_pred: {fp.shape}')

In [ ]:
# === 6. Loss + Lightning Module ===
class FocalBCELoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, pred, target):
        if self.smoothing > 0:
            nc = target.shape[-1]
            target = target * (1 - self.smoothing) + self.smoothing * (target.sum(dim=-1, keepdim=True) / nc)
        bce = F.binary_cross_entropy(pred, target, reduction='none')
        pt = torch.where(target > 0.5, pred, 1 - pred)
        focal = (1 - pt) ** self.gamma * bce
        return 0.5 * bce.mean() + 0.5 * (self.alpha * focal).mean()

class BirdModule(pl.LightningModule):
    def __init__(self, backbone='tf_efficientnet_b0_ns', nc=234, lr=1e-4, epochs=15):
        super().__init__()
        self.save_hyperparameters()
        self.model = SEDModel(backbone, nc)
        self.criterion = FocalBCELoss()
        self.val_outputs = []
        self.lr = lr
        self.epochs = epochs

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        mel, target = batch
        # MixUp (element-wise max targets, +3.6% AUC from 2025 2nd place)
        if np.random.random() < 0.5:
            idx = torch.randperm(mel.size(0), device=mel.device)
            lam = np.random.beta(0.4, 0.4)
            mel = lam * mel + (1 - lam) * mel[idx]
            target = torch.max(target, target[idx])
        clip_p, _ = self.model(mel)
        loss = self.criterion(clip_p, target)
        self.log('train/loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        mel, target = batch
        clip_p, _ = self.model(mel)
        loss = self.criterion(clip_p, target)
        self.val_outputs.append({'preds': clip_p.cpu(), 'targets': target.cpu()})
        self.log('val/loss', loss, prog_bar=True)

    def on_validation_epoch_end(self):
        preds = torch.cat([o['preds'] for o in self.val_outputs]).numpy()
        targets = torch.cat([o['targets'] for o in self.val_outputs]).numpy()
        aucs = []
        for i in range(targets.shape[1]):
            if targets[:, i].sum() > 0:
                try:
                    aucs.append(roc_auc_score(targets[:, i], preds[:, i]))
                except: pass
        macro_auc = np.mean(aucs) if aucs else 0.0
        self.log('val/macro_auc', macro_auc, prog_bar=True)
        print(f'\n>>> Epoch {self.current_epoch} — val/macro_auc: {macro_auc:.4f} ({len(aucs)}/{targets.shape[1]} classes)')
        self.val_outputs.clear()

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=0.01)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.epochs, eta_min=1e-6)
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sch, 'interval': 'epoch'}}

print('Module ready.')

In [ ]:
# === 7. Train Fold 0 ===
FOLD = 0
EPOCHS = 15
BATCH_SIZE = 32

train_meta = train_df[train_df['fold'] != FOLD]
val_meta = train_df[train_df['fold'] == FOLD]
print(f'Train: {len(train_meta)}, Val: {len(val_meta)}')

train_ds = BirdDataset(train_meta, AUDIO_DIR, train=True)
val_ds = BirdDataset(val_meta, AUDIO_DIR, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True)

module = BirdModule('tf_efficientnet_b0_ns', num_classes, lr=1e-4, epochs=EPOCHS)

OUT = '/content/models/fold_0'
os.makedirs(OUT, exist_ok=True)

ckpt_cb = ModelCheckpoint(dirpath=OUT,
    filename='best-{epoch:02d}-{val/macro_auc:.4f}',
    monitor='val/macro_auc', mode='max', save_top_k=1, save_last=True)
early_cb = EarlyStopping(monitor='val/macro_auc', mode='max', patience=5, verbose=True)

trainer = pl.Trainer(
    max_epochs=EPOCHS, callbacks=[ckpt_cb, early_cb],
    default_root_dir=OUT, precision='16-mixed',
    accelerator='gpu', devices=1, log_every_n_steps=10)

trainer.fit(module, train_loader, val_loader)

print(f'\n{"="*60}')
print(f'BEST val/macro_auc: {ckpt_cb.best_model_score:.4f}')
print(f'Best checkpoint: {ckpt_cb.best_model_path}')
print(f'SCORE: {ckpt_cb.best_model_score:.6f}')
print(f'{"="*60}')

In [ ]:
# === 8. Save to Drive ===
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/birdclef-2026/models/
    !cp -r /content/models/fold_0/ /content/drive/MyDrive/birdclef-2026/models/fold_0/
    print('Saved to Google Drive!')
except Exception as e:
    print(f'Drive save failed: {e}')
    print('Download checkpoint manually from /content/models/fold_0/')